# XYZ 分子交互可视化

使用 `py3Dmol` 在 Notebook 内直接旋转、缩放和查看 XYZ 分子，不生成 HTML 文件。通过“起始行”和“数量 n”控制每次加载的 XYZ；默认从第 0 行开始显示 5 个分子，单次最多显示 20 个。

In [1]:
from pathlib import Path
import sys

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, clear_output, display


def find_project_root(start: Path) -> Path:
    """从当前 Jupyter 工作目录向上寻找项目根目录。"""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "cof_symmetry_pipeline/output/final_dataset.csv").is_file():
            return candidate
    raise FileNotFoundError("未找到 MARL_for_COFs 项目根目录，请从项目目录启动 Jupyter")


PROJECT_ROOT = find_project_root(Path.cwd())
VIS_DIR = PROJECT_ROOT / "cof_symmetry_pipeline/visualization"
if str(VIS_DIR) not in sys.path:
    sys.path.insert(0, str(VIS_DIR))

from visualize_xyz import build_viewer, parse_xyz

CSV_PATH = PROJECT_ROOT / "cof_symmetry_pipeline/output/final_dataset.csv"
dataset = pd.read_csv(CSV_PATH)
print(f"项目根目录: {PROJECT_ROOT}")
print(f"已加载 {len(dataset):,} 个分子")

项目根目录: /home/tianyajun/MARL_for_COFs
已加载 2,532 个分子


In [2]:
def resolve_xyz_path(csv_path: str) -> Path:
    """优先使用 CSV 路径；项目移动后按 cof_symmetry_pipeline 后缀重建路径。"""
    path = Path(csv_path)
    if path.is_file():
        return path
    try:
        marker_index = path.parts.index("cof_symmetry_pipeline")
    except ValueError as exc:
        raise FileNotFoundError(f"XYZ 文件不存在: {path}") from exc
    relocated = PROJECT_ROOT.joinpath(*path.parts[marker_index:])
    if not relocated.is_file():
        raise FileNotFoundError(f"XYZ 文件不存在: {path}；重定位后仍不存在: {relocated}")
    return relocated


def show_xyz(path, style="ball-and-stick", labels=False, spin=False, background="white"):
    """在当前 Notebook 输出区内显示一个 XYZ 分子。"""
    path = Path(path)
    xyz_text, atoms = parse_xyz(path)
    viewer = build_viewer(
        xyz_text=xyz_text,
        atoms=atoms,
        style=style,
        background=background,
        labels=labels,
        spin=spin,
    )
    display(viewer)
    return viewer


def show_dataset_batch(start_index, n_files, style, labels, spin, background):
    """只读取并显示 [start_index, start_index + n_files) 范围内的 XYZ。"""
    stop_index = min(start_index + n_files, len(dataset))
    display(Markdown(
        f"本次显示数据行 **{start_index}–{stop_index - 1}**，"
        f"共 **{stop_index - start_index}** 个 XYZ。"
    ))
    for row_index in range(start_index, stop_index):
        row = dataset.iloc[row_index]
        xyz_path = resolve_xyz_path(row["XYZ_Path"])
        display(Markdown(f"### 数据行 {row_index}: `{xyz_path.name}`"))
        metadata = row[[
            "SMILES", "Core", "Arm", "Target_PG", "Point_Group",
            "Geometry", "Energy_kcal_mol", "Source",
        ]].to_frame().T
        metadata.index = [row_index]
        display(metadata)
        show_xyz(xyz_path, style=style, labels=labels, spin=spin, background=background)


In [3]:
start_control = widgets.BoundedIntText(
    value=0, min=0, max=len(dataset) - 1, step=1, description="起始行:"
)
count_control = widgets.BoundedIntText(
    value=1, min=1, max=20, step=1, description="数量 n:"
)
style_control = widgets.Dropdown(
    options=["ball-and-stick", "stick", "sphere", "line"],
    value="ball-and-stick", description="样式:"
)
label_control = widgets.Checkbox(value=False, description="显示非氢元素标签")
spin_control = widgets.Checkbox(value=False, description="自动旋转")
background_control = widgets.ColorPicker(value="#ffffff", description="背景:")
render_button = widgets.Button(description="显示这 n 个 XYZ", button_style="primary")
output = widgets.Output()

controls = widgets.HBox([
    widgets.VBox([start_control, count_control, style_control]),
    widgets.VBox([label_control, spin_control, background_control, render_button]),
])

def render_batch(_=None):
    with output:
        clear_output(wait=True)
        show_dataset_batch(
            start_index=start_control.value,
            n_files=count_control.value,
            style=style_control.value,
            labels=label_control.value,
            spin=spin_control.value,
            background=background_control.value,
        )

render_button.on_click(render_batch)
display(controls, output)
render_batch()

Output()

## 查看任意 XYZ（可选）

若文件不在 `final_dataset.csv` 中，可新建一个代码单元并调用：

```python
show_xyz("/path/to/molecule.xyz", style="ball-and-stick", labels=False)
```